# Counts and metadata

The other notebooks stop at the metadata. This one goes the rest of the way:
from an accession to a counts matrix, with the per-cell annotation the
submitter deposited and the study metadata seqout.org holds, joined into one
object you can hand to scanpy.

Reading counts needs the `counts` extra:

```bash
uv add 'seqout[counts]'
```

`.rds` files additionally need R on the PATH, with whichever package wrote the
object. That is only a fallback; most `.rds` files are read without it.

In [1]:
import pandas as pd

from seqout import connect, seqout_counts

ACCESSION = "GSE297547"  # PBMCs from myasthenia gravis patients and controls

## Fetching counts form GEO 

`seqout_counts` resolves and groups files in the constructor but downloads
nothing. GEO supplementary payloads run to tens of gigabytes, so the manifest
comes first and costs only the API calls that list the files.

A unit is a group of files that read as one matrix: a 10x triplet
(`matrix.mtx` + `barcodes.tsv` + `features.tsv`), a CellRanger `.h5`, an
`.h5ad`, an `.rds`, or a delimited table.

In [2]:
counts = seqout_counts(gse=ACCESSION)
manifest = counts.manifest()

print(f"{len(manifest)} units")
manifest[["unit", "sample", "format", "preferred", "has_metadata"]].head(8)

39 units


,unit,sample,format,preferred,has_metadata
0,GSM8994520:adt_C001,GSM8994520,h5ad,False,True
1,GSM8994520:rna_C001,GSM8994520,h5ad,True,True
2,GSM8994521:adt_C002,GSM8994521,h5ad,False,True
3,GSM8994521:rna_C002,GSM8994521,h5ad,True,True
4,GSM8994522:adt_C003,GSM8994522,h5ad,False,True
5,GSM8994522:rna_C003,GSM8994522,h5ad,True,True
6,GSM8994523:adt_C004,GSM8994523,h5ad,False,True
7,GSM8994523:rna_C004,GSM8994523,h5ad,True,True


`preferred` marks the unit that `matrix()` picks for each sample. A submitter
often deposits the same counts twice, for instance a 10x `.h5` alongside the
mtx triplet it was built from. Both are listed; the losing one stays reachable
by its label so nothing is hidden from you.

`has_metadata` says whether per-cell annotation is available at all, either as
a sidecar file or embedded in the container. `.h5ad` and `.rds` always carry
their own, which is why every unit here reports `True`.

In [3]:
manifest["format"].value_counts()

format
h5ad    38
tar      1
Name: count, dtype: int64

## Reading one sample

Picking a single unit keeps this notebook fast. `matrix()` downloads that
unit's files, caches them under `~/.cache/seqout/counts/`, and reads them.

In [4]:
m = counts.matrix(sample=counts.units()[0].label)
m

CountMatrix(GSM8994520, single_cell, h5ad, 11032 cells x 25259 genes)

`CountMatrix` is deliberately in AnnData orientation: rows are observations,
columns are genes. For single-cell data a row is a cell; for bulk it is a
biological sample. That is what lets one type cover both without the caller
checking which they have.

In [5]:
m.summary

accession                           GSM8994520
kind                               single_cell
format                                    h5ad
observations                             11032
features                                 25259
sparse                                    True
obs_columns                                  8
metadata_fields    sample_id, tissue, sex, age
evidence                             h5ad file
source                GSM8994520_rna_C001.h5ad
Name: GSM8994520, dtype: object

`kind` is decided on evidence. Barcode-looking row
labels mean single-cell; row labels that are GSM accessions, or an observation
count no larger than the number of samples in the series, mean bulk. A dense
table with a few hundred columns could be a Smart-seq plate or a large bulk
study, and in that band `kind` reports `unknown`. Check
`evidence` when the answer matters.

## Cell-level metadata

`obs` carries whatever per-cell annotation the submitter deposited. For an
`.h5ad` or `.rds` that comes from inside the container; for a 10x unit it comes
from a sidecar file, joined on the cell label.

In [6]:
m.obs.head()

,type1,type2,type3,title,tissue,age,Sex,sample
C001_13,Control,WTA,C001,PBMC from C01 sample,PBMC,45,male,GSM8994520
C001_64,Control,WTA,C001,PBMC from C01 sample,PBMC,45,male,GSM8994520
C001_84,Control,WTA,C001,PBMC from C01 sample,PBMC,45,male,GSM8994520
C001_398,Control,WTA,C001,PBMC from C01 sample,PBMC,45,male,GSM8994520
C001_800,Control,WTA,C001,PBMC from C01 sample,PBMC,45,male,GSM8994520


Column names differ wildly between submitters, so `metadata_fields` maps them
onto a fixed vocabulary. It records the source columns, not just a flag: a
`condition` derived from `disease_status` is not the same variable as one
derived from `treatment`, and you usually need to know which you got.

In [7]:
m.metadata_fields

{'sample_id': ['sample'], 'tissue': ['tissue'], 'sex': ['Sex'], 'age': ['age']}

Empty, and that is the honest answer. This submitter named their annotation
`type1`, `type2`, `type3`, which carries no vocabulary a matcher can key on.
The columns are still there in `obs` and still usable; what you do not get is
a claim that a particular one is the cell type.

When the names are recognisable the mapping fills in. Clusters are kept
deliberately separate from cell types: `seurat_clusters`, `leiden` and
`louvain` are numeric labels, not biology, and counting them as annotation
would overstate how many studies carry real cell-type calls.

In [8]:
m.obs["type1"].value_counts().head()

type1
Control    11032
Name: count, dtype: int64

## Picking an assay

This is a CITE-seq study, so every sample ships two matrices: gene expression
and an antibody panel. Both are `.h5ad`, so nothing in the file type separates
them, and the antibody file happens to sort first alphabetically.

`seqout_counts` defaults to `assay="rna"` for that reason. Ask for the panel
explicitly and you get it, with its own annotation.

In [9]:
adt = seqout_counts(gse=ACCESSION, assay="adt")
adt_m = adt.matrix(sample=adt.units()[0].label)

pd.DataFrame({"rna": m.summary, "adt": adt_m.summary})

,rna,adt
accession,GSM8994520,GSM8994520
kind,single_cell,single_cell
format,h5ad,h5ad
observations,11032,11032
features,25259,9
sparse,True,True
obs_columns,8,56
metadata_fields,"sample_id, tissue, sex, age","sample_id, celltype, tissue, sex, age"
evidence,h5ad file,h5ad file
source,GSM8994520_rna_C001.h5ad,GSM8994520_adt_C001.h5ad


In [10]:
adt_m.obs["Cell_Type_Experimental"].value_counts().head()

Cell_Type_Experimental
B                 5207
T_CD4_memory      4860
T_CD8_memory       890
Natural_killer      70
T_gamma_delta        4
Name: count, dtype: int64

In [11]:
adt.close()

## Study metadata from seqout.org

The counts tell you what was measured. They say nothing about who the donors
were, what the condition was, or which paper this came from. That lives in the
API.

In [12]:
sq = connect("api")
project = sq.fetch_project_metadata(ACCESSION)

print(project.title)
print()
print("organisms :", project.organisms)
print("published :", project.published_at)
print("modality  :", project.single_cell_modality)
print("samples   :", len(project.samples_ref))

Single-cell RNA sequencing of PBMCs from patients with myasthenia gravis and healthy controls

organisms : ['Homo sapiens']
published : 2025-06-30
modality  : scRNA-seq
samples   : 25


Sample-level characteristics are where the experimental design lives, and
`counts.design` returns them as a frame indexed by accession. GEO stores them
as free-text key/value pairs chosen by the submitter, so the columns vary by
study and are worth inspecting before you rely on them. The same frame is
available for any sample list via `seqout.sample_frame(...)`.

In [13]:
counts.design.head()

,title,tissue,age,Sex
sample,,,,
GSM8994520,PBMC from C01 sample,PBMC,45,male
GSM8994521,PBMC from C02 sample,PBMC,55,male
GSM8994522,PBMC from C03 sample,PBMC,63,female
GSM8994523,PBMC from C04 sample,PBMC,57,female
GSM8994524,PBMC from C05 sample,PBMC,70,female


## Fetching all samples counts

There is nothing to join by hand. Every cell in a unit came from one GSM, so
`matrix()` already carried that sample's characteristics onto `obs` when it
read the unit. Per-cell annotation wins where both define a column, since it
is the more specific of the two.

In [14]:
m.obs[["type1", "sample", "tissue", "age", "Sex"]].head()

,type1,sample,tissue,age,Sex
C001_13,Control,GSM8994520,PBMC,45,male
C001_64,Control,GSM8994520,PBMC,45,male
C001_84,Control,GSM8994520,PBMC,45,male
C001_398,Control,GSM8994520,PBMC,45,male
C001_800,Control,GSM8994520,PBMC,45,male


The AnnData is complete as soon as it is built.

In [15]:
adata = m.to_anndata()
adata

AnnData object with n_obs × n_vars = 11032 × 25259
    obs: 'type1', 'type2', 'type3', 'title', 'tissue', 'age', 'Sex', 'sample'
    var: 'Raw_Reads-0', 'Raw_Molecules-0', 'Raw_Seq_Depth-0', 'RSEC_Adjusted_Molecules-0', 'RSEC_Adjusted_Reads_non-singleton-0', 'RSEC_Adjusted_Molecules_non-singleton-0', 'Raw_Reads-1', 'Raw_Molecules-1', 'Raw_Seq_Depth-1', 'RSEC_Adjusted_Molecules-1', 'RSEC_Adjusted_Reads_non-singleton-1', 'RSEC_Adjusted_Molecules_non-singleton-1', 'Raw_Reads-10', 'Raw_Molecules-10', 'Raw_Seq_Depth-10', 'RSEC_Adjusted_Molecules-10', 'RSEC_Adjusted_Reads_non-singleton-10', 'RSEC_Adjusted_Molecules_non-singleton-10', 'Raw_Reads-11', 'Raw_Molecules-11', 'Raw_Seq_Depth-11', 'RSEC_Adjusted_Molecules-11', 'RSEC_Adjusted_Reads_non-singleton-11', 'RSEC_Adjusted_Molecules_non-singleton-11', 'Raw_Reads-12', 'Raw_Molecules-12', 'Raw_Seq_Depth-12', 'RSEC_Adjusted_Molecules-12', 'RSEC_Adjusted_Reads_non-singleton-12', 'RSEC_Adjusted_Molecules_non-singleton-12', 'Raw_Reads-13', 'Raw_Mol

From here it is an ordinary AnnData: `sc.pp.filter_cells`, `sc.pp.normalize_total`
and the rest of the scanpy pipeline all work, and the donor covariates you just
attached are available for grouping and for regressing out batch.

The provenance of the matrix is recorded in `uns` so a downstream reader can
tell where the numbers came from.

In [16]:
adata.uns["seqout"]

{'accession': 'GSM8994520',
 'kind': 'single_cell',
 'format': 'h5ad',
 'source': 'GSM8994520_rna_C001.h5ad',
 'evidence': ['h5ad file'],
 'has_metadata': True,
 'metadata_fields': {'sample_id': ['sample'],
  'tissue': ['tissue'],
  'sex': ['Sex'],
  'age': ['age']}}

## Whole studies, and other formats

`matrices()` reads every preferred unit and returns a dict keyed by unit label.
It prefetches all the files in one threaded call, because downloading a
25-sample series one stream at a time is the slow way to do it.

```python
mats = counts.matrices()               # every sample
adata = counts.anndata()               # concatenated, inner join on genes
paths = counts.raw(sample="GSM...")    # just the files, no parsing
```

The download is the expensive part, and a series can be tens of gigabytes.
`matrices()` skips a unit it cannot read, so compare the number of results
against the manifest.

In [17]:
counts.close()
sq.close()

## When metadata does not line up

`has_metadata` on a unit means an annotation file exists. It does not promise
the file describes these cells. Submitters routinely deposit a series-level
metadata table whose row labels do not match the barcodes in any individual
matrix.

When the labels do not overlap, the join is skipped and logged, because an
outer join would fill `obs` with a column of nulls that looks like real
annotation. `CountMatrix.has_metadata` reports what actually
landed, which is why it can be `False` on a unit whose manifest row says
`True`. Turn on `INFO` logging to see the skip:

```python
import logging
logging.basicConfig(level=logging.INFO)
```